# AgentCore Harness와 Agent Skills 통합

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | Harness - 기능 확장을 위한 Agent Skills |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

**학습 내용:**
- **Agent Skills**의 개념과 중요성
- 에이전트 VM에 skill을 설치하는 방법
- 호출 시 `skills` 파라미터 사용
- 파일 형식 Agent Skills(xlsx, pdf, docx) 활용
- Agent Skills의 도움을 받아 복잡한 결과 생성
- Agent Skills 관리 모범 사례

**Agent Skills란 무엇인가요?**

Agent Skills는 다음 항목을 통해 에이전트의 기능을 확장하는 사전 구축된 기능 묶음입니다.
- **전문 지침** - 복잡한 작업을 위한 단계별 안내
- **코드 템플릿** - 파일 형식, API 등에 대해 검증된 구현
- **도구 설정** - 사전 구성된 설정 및 예제
- **도메인 지식** - 모범 사례 및 일반적인 패턴

Agent Skills는 전문 파일 형식이나 API에 대한 내장 지식이 부족한 소형 또는 저비용 모델에서 특히 유용합니다. 모델이 작업을 성공적으로 수행하는 데 필요한 지침과 템플릿을 제공합니다.

**사용 가능한 Agent Skills:**
- `xlsx` - Excel 스프레드시트 생성 및 조작
- `pdf` - PDF 생성 및 처리
- `docx` - Word document 생성
- GitHub 또는 로컬 디렉터리의 사용자 지정 Agent Skills

**사전 요구 사항:**
- Amazon Bedrock AgentCore에 접근할 수 있는 AWS 계정
- 자격 증명이 설정된 AWS CLI v2
- Python 3.10+

## Part 0: 설정

헬퍼 모듈을 가져오고 IAM 실행 역할을 생성합니다.

In [ ]:
import sys
import os
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## Part 1: Harness 생성

모든 Agent Skills 예제에서 사용할 표준 Harness를 생성합니다.

In [ ]:
HARNESS_NAME = f"SkillsDemo_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(harnessName=HARNESS_NAME, executionRoleArn=role_arn)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")
print(f"Status: {harness['status']}")

### Harness가 준비될 때까지 대기

In [ ]:
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness is ready")
        break
    time.sleep(5)

Agent Skills를 설치할 수 있도록 Node 이미지로 컨테이너 업데이트

In [ ]:
CONTAINER_URI = "public.ecr.aws/docker/library/node:slim"

control.update_harness(
    harnessId=harness_id,
    environmentArtifact={"optionalValue": {"containerConfiguration": {"containerUri": CONTAINER_URI}}},
)

In [ ]:
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness is ready")
        break
    time.sleep(5)

## Part 2: Agent Skills 설치

`ExecuteCommand`를 사용하여 에이전트 VM에 Agent Skills를 설치합니다. Anthropic skills 저장소는 설치를 처리하는 CLI 도구를 제공합니다.

**설치 패턴:**
```bash
npx skills add <github-url> --skill <skill-name> --yes
```

이 명령은 Agent Skill을 내려받아 VM의 `.agents/skills/<skill-name>`에 설치합니다.

Excel 스프레드시트 생성을 위한 `xlsx` Agent Skill을 설치하겠습니다.

In [ ]:
session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

# Anthropic skills 저장소에서 xlsx Agent Skill 설치
print("Installing xlsx skill...")
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=session_id,
    body={
        "command": "apt-get update && apt-get install git -y && npx skills add https://github.com/anthropics/skills --skill xlsx --yes"
    },
)

output = ""
for event in resp["stream"]:
    if "chunk" in event and "contentDelta" in event["chunk"]:
        delta = event["chunk"]["contentDelta"]
        if "stdout" in delta:
            output += delta["stdout"]
        if "stderr" in delta:
            output += delta["stderr"]

# 설치 출력이 길 수 있으므로 마지막 500자 표시
print(output[-500:] if len(output) > 500 else output)
print("\n✅ xlsx skill installed")

### Agent Skill 설치 확인

In [ ]:
# Agent Skill 디렉터리가 있는지 확인
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=session_id,
    body={"command": "ls -la .agents/skills/xlsx/"},
)

for event in resp["stream"]:
    if "chunk" in event and "contentDelta" in event["chunk"]:
        delta = event["chunk"]["contentDelta"]
        if "stdout" in delta:
            print(delta["stdout"], end="")

## Part 3: 호출에서 Agent Skills 사용

Agent Skill을 사용하려면 에이전트를 호출할 때 `skills` 파라미터로 전달합니다.

```python
skills=[{"path": ".agents/skills/xlsx"}]
```

에이전트는 Agent Skill의 지침을 자동으로 불러와 작업을 완료하는 데 사용합니다.

### 예제: 여행 예산 스프레드시트 생성

에이전트에 여행 예산이 포함된 Excel 스프레드시트를 생성하도록 요청해 보겠습니다. xlsx Agent Skill을 불러오면 에이전트는 다음 내용을 알 수 있습니다.
- xlsx 파일을 구성하는 방법
- 사용할 npm 패키지
- 수식과 서식의 일반적인 패턴

In [ ]:
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    skills=[{"path": ".agents/skills/xlsx"}],
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Create an Excel spreadsheet with a 5-day Amsterdam trip budget. "
                        "Include columns for: Day, Category, Item, Cost (EUR), Cost (USD). "
                        "Add rows for accommodation, food, transport, museums, and activities for each day. "
                        "Include a total row with SUM formulas for EUR and USD columns. "
                        "Use currency conversion rate: 1 EUR = 1.10 USD. "
                        "Apply nice formatting: bold headers, currency formatting, alternating row colors. "
                        "Save it as /tmp/amsterdam_budget.xlsx"
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

# 응답 스트리밍
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            tool_name = start["toolUse"].get("name", "?")
            print(f"\n[Tool: {tool_name}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print("\n")

### 스프레드시트 내려받기 및 점검

In [ ]:
import base64

# 에이전트 VM에서 파일을 base64로 읽기
b64_data = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=session_id,
    body={"command": "base64 /tmp/amsterdam_budget.xlsx"},
)

for event in resp["stream"]:
    if "chunk" in event and "contentDelta" in event["chunk"]:
        delta = event["chunk"]["contentDelta"]
        if "stdout" in delta:
            b64_data += delta["stdout"]

# 로컬에 저장
if b64_data.strip():
    local_path = os.path.expanduser("~/Downloads/amsterdam_budget.xlsx")
    with open(local_path, "wb") as f:
        f.write(base64.b64decode(b64_data))
    print(f"✅ Saved to {local_path}")
    print(f"   Size: {os.path.getsize(local_path):,} bytes")
    print(f"\n   Open it with: open {local_path}")
else:
    print("⚠️  No file generated - check the agent's response above")

## Part 4: 여러 Agent Skills 설치

같은 VM에 여러 Agent Skills를 설치하고 함께 사용할 수 있습니다. 자주 사용하는 Agent Skills를 몇 개 더 설치해 보겠습니다.

**참고:** 이 예제는 시연용이므로 이 노트북에서 모든 Agent Skills를 사용하지는 않지만, 여러 Agent Skills를 사용하는 에이전트의 구축 패턴을 보여 줍니다.

In [ ]:
# 추가 Agent Skills 설치(선택 사항, 1분 정도 걸릴 수 있음)
multi_skill_session = str(uuid.uuid4()).upper()
print(f"Multi-skill session: {multi_skill_session}\n")

# 같은 저장소에서 여러 Agent Skills를 설치할 수 있음
skills_to_install = [
    # "pdf",    # PDF 생성
    # "docx",   # Word 문서
]

for skill_name in skills_to_install:
    print(f"Installing {skill_name} skill...")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=multi_skill_session,
        body={"command": f"npx skills add https://github.com/anthropics/skills --skill {skill_name} --yes"},
    )

    output = ""
    for event in resp["stream"]:
        if "chunk" in event and "contentDelta" in event["chunk"]:
            delta = event["chunk"]["contentDelta"]
            if "stdout" in delta:
                output += delta["stdout"]

    print(f"  ✅ {skill_name} installed\n")

if not skills_to_install:
    print("(Skipped - uncomment skills_to_install to try this)")

### 한 번의 호출에서 여러 Agent Skills 사용

여러 Agent Skills가 설치되어 있으면 여러 경로를 전달하여 모두 불러올 수 있습니다.

```python
skills=[
    {"path": ".agents/skills/xlsx"},
    {"path": ".agents/skills/pdf"},
    {"path": ".agents/skills/docx"},
]
```

에이전트는 작업에 따라 적절한 Agent Skill을 사용합니다.

## Part 5: 고급 예제 - 재무 보고서

여러 시트, 수식 및 서식이 포함된 더 복잡한 스프레드시트를 만들어 보겠습니다.

Agent Skills가 상세한 지침을 제공하므로 소형 모델도 정교한 Excel 파일을 생성할 수 있음을 보여 줍니다.

In [ ]:
print(f"Report session: {session_id}\n")

# 이전 세션에서 보고서 생성
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    skills=[{"path": ".agents/skills/xlsx"}],
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": (
                        "Create a professional quarterly sales report Excel file with 3 sheets:\n"
                        "\n"
                        "Sheet 1 - Summary:\n"
                        "- Q1 2024 Sales Overview title\n"
                        "- Table with: Region, Target (USD), Actual (USD), Variance (%), Status\n"
                        "- 4 regions: North America, Europe, Asia Pacific, Latin America\n"
                        "- Use realistic numbers (targets 1M-5M, actual should vary ±20%)\n"
                        "- Variance formula: (Actual-Target)/Target * 100\n"
                        "- Status formula: IF variance >= 0, 'On Track', 'Below Target'\n"
                        "- Total row with SUM formulas\n"
                        "\n"
                        "Sheet 2 - Monthly Breakdown:\n"
                        "- Table with: Month, Region, Sales (USD)\n"
                        "- Data for Jan, Feb, Mar for each region\n"
                        "- Total row\n"
                        "\n"
                        "Sheet 3 - Top Products:\n"
                        "- Table with: Rank, Product, Category, Units Sold, Revenue (USD)\n"
                        "- 10 products with realistic data\n"
                        "\n"
                        "Formatting:\n"
                        "- Bold headers with background color\n"
                        "- Currency formatting for USD columns\n"
                        "- Percentage formatting for Variance column\n"
                        "- Conditional formatting: green for positive variance, red for negative\n"
                        "- Freeze top row in all sheets\n"
                        "\n"
                        "Save as /tmp/q1_sales_report.xlsx"
                    )
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

# 응답 스트리밍
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            tool_name = start["toolUse"].get("name", "?")
            print(f"\n[Tool: {tool_name}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print("\n")

### 영업 보고서 내려받기

In [ ]:
# 보고서 내려받기
b64_data = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=session_id,
    body={"command": "base64 /tmp/q1_sales_report.xlsx 2>/dev/null"},
)

for event in resp["stream"]:
    if "chunk" in event and "contentDelta" in event["chunk"]:
        delta = event["chunk"]["contentDelta"]
        if "stdout" in delta:
            b64_data += delta["stdout"]

if b64_data.strip():
    local_path = os.path.expanduser("~/Downloads/q1_sales_report.xlsx")
    with open(local_path, "wb") as f:
        f.write(base64.b64decode(b64_data))
    print(f"✅ Saved to {local_path}")
    print(f"   Size: {os.path.getsize(local_path):,} bytes")
    print("   Sheets: 3 (Summary, Monthly Breakdown, Top Products)")
    print(f"\n   Open it with: open {local_path}")
else:
    print("⚠️  No file generated")

## 요약

다음 내용을 학습했습니다.
- ✅ npx를 사용하여 VM에 Agent Skills 설치
- ✅ 호출 시 `skills` 파라미터로 Agent Skills 사용
- ✅ xlsx Agent Skill로 복잡한 Excel 스프레드시트 생성
- ✅ 생성된 파일 내려받기 및 점검
- ✅ 서로 다른 기능을 위한 여러 Agent Skills 설치

### 다음 단계
1. [Anthropic skills 저장소](https://github.com/anthropics/skills)의 다른 Agent Skills 사용
2. 조직의 특정 요구 사항에 맞는 사용자 지정 Agent Skills 생성
3. Agent Skills를 MCP, Browser 및 Code Interpreter 도구와 조합
4. 여러 세션에 걸쳐 Agent Skills를 사용하는 다단계 워크플로 구축

## 리소스 정리

작업을 마치면 요금이 발생하지 않도록 Harness를 삭제합니다.

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
# IAM 역할 삭제(다른 예제에서 사용할 수 있으므로, 선택 사항)
delete_harness_role()
print("Deleted IAM role")